In [1]:
import asyncio
from decimal import Decimal
from datetime import datetime, timedelta, timezone

from typed_bit2me import Bit2Me
from dotenv import load_dotenv

from tribulnation.sdk.reporting import (
  CryptoDeposit,
  CryptoWithdrawal,
  Fee,
  HistoryRecord,
  Snapshot,
  SnapshotRecord,
  SpotTrade,
  SubaccountSnapshot,
  Yield,
  source_id,
)

load_dotenv()

client = await Bit2Me.new().__aenter__()

SPOT_MARKETS = ['BTC/EUR', 'ETH/EUR', 'USDT/EUR']

## `Snapshots`

Bit2Me splits balances across three separate sub-products with no shared ledger: the
Trading Spot wallet (`v1.trading.balance`), Earn (`v2.earn.wallets`), and Wallet pockets
(`v1.wallet.pockets.get`). Funds move between them only through explicit transfers
(`v1/trading/wallet/{deposit,withdraw}` between spot and pockets; `v1/earn/movements`
between earn and pockets), so they're not fungible views of one balance -- each becomes
its own `SubaccountSnapshot`.

In [2]:
async def spot_balances(*, assets: list[str] | None = None) -> dict[str, Decimal]:
  out: dict[str, Decimal] = {}
  for entry in await client.v1.trading.balance():
    asset = entry.get('currency')
    if asset is None or (assets is not None and asset not in assets):
      continue
    total = Decimal(str(entry.get('balance', 0))) + Decimal(
      str(entry.get('blockedBalance', 0))
    )
    out[asset] = out.get(asset, Decimal(0)) + total
  return out


await spot_balances()

{'BTC': Decimal('0.0'),
 'B2M': Decimal('0.0'),
 'EURC': Decimal('0.0'),
 'EURR': Decimal('1E-7'),
 'EUR': Decimal('0.00769999'),
 'USDC': Decimal('0.0')}

In [3]:
async def earn_balances(*, assets: list[str] | None = None) -> dict[str, Decimal]:
  out: dict[str, Decimal] = {}
  resp = await client.v2.earn.wallets(limit=100)
  for entry in resp.get('data', []):
    asset = entry.get('currency')
    if asset is None or (assets is not None and asset not in assets):
      continue
    out[asset] = out.get(asset, Decimal(0)) + Decimal(str(entry.get('balance', 0)))
  return out


await earn_balances()

{'B2M': Decimal('3391.41664338'),
 'EURC': Decimal('0E-8'),
 'EURR': Decimal('0E-8')}

In [4]:
async def pocket_balances(*, assets: list[str] | None = None) -> dict[str, Decimal]:
  out: dict[str, Decimal] = {}
  for entry in await client.v1.wallet.pockets.get():
    asset = entry.get('currency')
    if asset is None or (assets is not None and asset not in assets):
      continue
    total = Decimal(str(entry.get('balance', 0))) + Decimal(
      str(entry.get('blockedBalance', 0))
    )
    out[asset] = out.get(asset, Decimal(0)) + total
  return out


await pocket_balances()

{'EUR': Decimal('0.00867987'),
 'WBTC': Decimal('0E-8'),
 'BTC': Decimal('0E-8'),
 'EURR': Decimal('0E-8'),
 'B2M': Decimal('0E-8'),
 'USDC': Decimal('0E-8'),
 'EURC': Decimal('0E-8'),
 'ETH': Decimal('0E-8')}

In [5]:
async def snapshot(assets: list[str] | None = None) -> SnapshotRecord:
  spot, earn, pocket = await asyncio.gather(
    spot_balances(assets=assets),
    earn_balances(assets=assets),
    pocket_balances(assets=assets),
  )
  return SnapshotRecord(
    snapshot=Snapshot(
      subaccounts=[
        SubaccountSnapshot(subaccount='spot', balances=spot),
        SubaccountSnapshot(subaccount='earn', balances=earn),
        SubaccountSnapshot(subaccount='pocket', balances=pocket),
      ]
    ),
    provenance={'source': 'api', 'service': 'bit2me', 'id': source_id('bit2me')},
  )


await snapshot()

SnapshotRecord(snapshot=Snapshot(time=datetime.datetime(2026, 9, 4, 18, 40, 57, 440000, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), subaccounts=[SubaccountSnapshot(subaccount='spot', balances={'BTC': Decimal('0.0'), 'B2M': Decimal('0.0'), 'EURC': Decimal('0.0'), 'EURR': Decimal('1E-7'), 'EUR': Decimal('0.00769999'), 'USDC': Decimal('0.0')}, positions={}), SubaccountSnapshot(subaccount='earn', balances={'B2M': Decimal('3391.41664338'), 'EURC': Decimal('0E-8'), 'EURR': Decimal('0E-8')}, positions={}), SubaccountSnapshot(subaccount='pocket', balances={'EUR': Decimal('0.00867987'), 'WBTC': Decimal('0E-8'), 'BTC': Decimal('0E-8'), 'EURR': Decimal('0E-8'), 'B2M': Decimal('0E-8'), 'USDC': Decimal('0E-8'), 'EURC': Decimal('0E-8'), 'ETH': Decimal('0E-8')}, positions={})]), provenance={'id': 'bit2me:2026-09-04T18:40:57.440025+00:00:4ac5b2c7-6310-46ee-b3dd-fd488e944391', 'source': 'api', 'service': 'bit2me'})

In [6]:
# `assets` is accepted per the abstract `Snapshots.snapshot()` signature but ignored by
# every helper above -- all three balance endpoints always enumerate every held asset in
# one call, so (per the abstract docstring, meant for venues without full enumeration)
# there's no discovery gap here for `assets` to fill.
await snapshot(assets=['BTC', 'ETH'])

SnapshotRecord(snapshot=Snapshot(time=datetime.datetime(2026, 9, 4, 10, 18, 37, 448595, tzinfo=datetime.timezone(datetime.timedelta(0), 'UTC')), subaccounts=[SubaccountSnapshot(subaccount='spot', balances={'ETH': Decimal('0.0')}, positions={}), SubaccountSnapshot(subaccount='earn', balances={}, positions={}), SubaccountSnapshot(subaccount='pocket', balances={'ETH': Decimal('0.00400000'), 'BTC': Decimal('0E-8')}, positions={})]), provenance={'id': 'bit2me:2026-09-04T10:18:37.448613+00:00:4e6005db-c0fc-49f2-ab7b-a512eda15321', 'source': 'api', 'service': 'bit2me'})

## `History`

No single Bit2Me endpoint is a unified ledger. This notebook maps three separately-shaped
sources, each hand-picked for having an unambiguous `Observation` mapping:

- **Spot trades**: `v1.trading.trade` (`client.v1.trading.trades.list`), per market symbol,
  natively filterable by `startTime`/`endTime`.
- **Crypto deposits/withdrawals**: `v2.wallet.transaction`
  (`client.v2.wallet.transactions`) filtered to `operation='receive'`/`'send'` -- *not*
  `'deposit'`/`'withdrawal'`, which select the fiat bank/card rows -- and then to rows
  where the blockchain side's `class` is `'blockchain'`. The only date filter is `year`,
  so the requested `start`/`end` window is walked one year at a time and re-applied
  client-side.
- **Earn rewards**: `v1.earn.wallets.list_movements`, one call per Earn wallet (there's no
  cross-wallet movements endpoint), filtered to `type == 'reward'`.

In [2]:
async def spot_trades(start: datetime, end: datetime) -> list[SpotTrade]:
  out: list[SpotTrade] = []
  for symbol in SPOT_MARKETS:
    base, quote = symbol.split('/')
    resp = await client.v1.trading.trades.list(
      symbol=symbol, start_time=start, end_time=end, limit=100
    )
    for t in resp.get('data', []):
      side = t.get('side')
      amount = Decimal(str(t.get('amount', 0)))
      size = amount if side == 'buy' else -amount
      price = t.get('price')
      fee_amount = t.get('feeAmount')
      fee_currency = t.get('feeCurrency')
      out.append(
        SpotTrade(
          id=t.get('id'),
          time=t.get('createdAt'),
          base=base,
          quote=quote,
          pair=symbol,
          size=size,
          price=Decimal(str(price)) if price is not None else None,
          order_id=t.get('orderId'),
          fee=Fee(amount=Decimal(str(fee_amount)), asset=fee_currency)
          if fee_amount and fee_currency
          else None,
        )
      )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(days=30)
await spot_trades(start, end)

[]

In [3]:
async def crypto_deposits(start: datetime, end: datetime) -> list[CryptoDeposit]:
  out: list[CryptoDeposit] = []
  # On-chain credits are `operation='receive'`, not `'deposit'`: `'deposit'`/`'withdrawal'`
  # select the fiat (bank/card) rows, and asking for `'deposit'` returns nothing at all on
  # this account. Only `year` is filterable server-side, so the window is walked one year
  # at a time and re-applied client-side afterwards.
  for year in range(start.year, end.year + 1):
    resp = await client.v2.wallet.transactions(
      operation='receive', limit=100, year=year
    )
    for tx in resp.get('data', []):
      origin = tx.get('origin') or {}
      dest = tx.get('destination') or {}
      time = tx.get('date')
      asset = dest.get('currency')
      if (
        origin.get('class') != 'blockchain'
        or time is None
        or asset is None
        or not (start <= time <= end)
      ):
        continue
      if tx.get('status') != 'completed':
        continue
      net_fee = (tx.get('fee') or {}).get('network')
      fee_amount = net_fee.get('amount') if net_fee else None
      fee_currency = net_fee.get('currency') if net_fee else None
      out.append(
        CryptoDeposit(
          id=tx.get('id'),
          time=time,
          asset=asset,
          amount=Decimal(str(dest.get('amount', 0))),
          # On a deposit the blockchain side is the *origin*, but it carries neither
          # `address` nor `addressNetwork` -- both sit on the crediting pocket instead.
          network=dest.get('addressNetwork'),
          tx_id=(tx.get('transaction') or {}).get('hash'),
          src_address=origin.get('address'),
          dst_address=dest.get('address'),
          fee=Fee(amount=Decimal(str(fee_amount)), asset=fee_currency)
          if fee_amount and fee_currency
          else None,
        )
      )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(days=365)
await crypto_deposits(start, end)

[CryptoDeposit(id='60f46a31-040f-447f-8d51-f080302a46eb', time=datetime.datetime(2025, 11, 3, 15, 30, 32, 166000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('295.48000000'), asset='USDC', network='binanceSmartChain', tx_id=None, src_address=None, dst_address='0x17b863e3f93Db771B819F6895ca2baADFbC73a13', fee=None, type='crypto_deposit'),
 CryptoDeposit(id='4add305d-79c2-4be4-b099-d3b65b450797', time=datetime.datetime(2025, 10, 11, 12, 26, 29, 503000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('11121.55000000'), asset='USDC', network='binanceSmartChain', tx_id=None, src_address=None, dst_address='0x17b863e3f93Db771B819F6895ca2baADFbC73a13', fee=None, type='crypto_deposit'),
 CryptoDeposit(id='30335152-dc7a-4f63-875a-711714b75c50', time=datetime.datetime(2025, 10, 11, 11, 44, 25, 220000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('11244.44000000'), asset='USDC', network='binanceSmartChain', tx_id=None, src_address=None, dst_addre

In [4]:
async def crypto_withdrawals(start: datetime, end: datetime) -> list[CryptoWithdrawal]:
  out: list[CryptoWithdrawal] = []
  # Mirror of `crypto_deposits`: on-chain debits are `operation='send'`. `'withdrawal'`
  # returns the fiat bank-transfer rows instead, which is why filtering by it and then
  # discarding everything whose destination isn't `'blockchain'` yields nothing.
  for year in range(start.year, end.year + 1):
    resp = await client.v2.wallet.transactions(operation='send', limit=100, year=year)
    for tx in resp.get('data', []):
      origin = tx.get('origin') or {}
      dest = tx.get('destination') or {}
      time = tx.get('date')
      asset = origin.get('currency')
      if (
        dest.get('class') != 'blockchain'
        or time is None
        or asset is None
        or not (start <= time <= end)
      ):
        continue
      # Cancelled withdrawals stay in the listing with the amount they would have sent.
      if tx.get('status') != 'completed':
        continue
      net_fee = (tx.get('fee') or {}).get('network')
      fee_amount = net_fee.get('amount') if net_fee else None
      fee_currency = net_fee.get('currency') if net_fee else None
      out.append(
        CryptoWithdrawal(
          id=tx.get('id'),
          time=time,
          asset=asset,
          amount=-Decimal(str(origin.get('amount', 0))),
          network=dest.get('addressNetwork'),
          tx_id=(tx.get('transaction') or {}).get('hash'),
          src_address=origin.get('address'),
          dst_address=dest.get('address'),
          fee=Fee(amount=Decimal(str(fee_amount)), asset=fee_currency)
          if fee_amount and fee_currency
          else None,
        )
      )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(days=365)
await crypto_withdrawals(start, end)

[CryptoWithdrawal(id='c92fc785-3ad4-4268-9937-5014240f4b3f', time=datetime.datetime(2025, 11, 6, 12, 12, 57, 943000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('-9996.59360000'), asset='EURC', network='ethereum', tx_id=None, src_address=None, dst_address='0x60ffe08b24dd61ec77080d6b60d86bde5333f01e', fee=None, type='crypto_withdrawal'),
 CryptoWithdrawal(id='60ce102b-d1c8-41e3-9a8b-364febf8623d', time=datetime.datetime(2025, 11, 6, 12, 10, 29, 932000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('-0.00994572'), asset='ETH', network='ethereum', tx_id=None, src_address=None, dst_address='0x60ffe08b24dd61ec77080d6b60d86bde5333f01e', fee=None, type='crypto_withdrawal'),
 CryptoWithdrawal(id='21ad339c-74a8-490f-a2fb-9c3410cf636f', time=datetime.datetime(2025, 10, 11, 11, 54, 10, 401000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('-9843.46996587'), asset='EURR', network='ethereum', tx_id=None, src_address=None, dst_address='0x86405ef1f

In [5]:
async def earn_yield(start: datetime, end: datetime) -> list[Yield]:
  wallets = await client.v2.earn.wallets(limit=100)
  out: list[Yield] = []
  for w in wallets.get('data', []):
    wallet_id = w.get('walletId')
    if wallet_id is None:
      continue
    movements = await client.v1.earn.wallets.list_movements(
      wallet_id=wallet_id,
      limit=50,
      sort_by='createdAt',
      sort_direction='descending',
    )
    for m in movements.get('data', []):
      time = m.get('createdAt')
      if m.get('type') != 'reward' or time is None or not (start <= time <= end):
        continue
      net = m.get('netAmount') or m.get('amount') or {}
      value = net.get('value')
      currency = net.get('currency')
      if value is None or currency is None:
        continue
      out.append(
        Yield(
          id=m.get('movementId'),
          time=time,
          asset=currency,
          amount=Decimal(str(value)),
        )
      )
  return out


end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
await earn_yield(start, end)

[Yield(id='80bb3a5e-43b3-44e1-a362-12d95f4a9258', time=datetime.datetime(2026, 9, 4, 11, 35, 41, 871000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('0.35782066'), asset='B2M', type='yield'),
 Yield(id='2a14c7fc-e4d9-4696-bb09-37f0df08b92b', time=datetime.datetime(2026, 9, 3, 11, 59, 48, 149000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('0.35814155'), asset='B2M', type='yield'),
 Yield(id='ff5ad206-db14-4945-9038-6c8b45bcea5b', time=datetime.datetime(2026, 9, 2, 11, 17, 3, 184000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('0.35737836'), asset='B2M', type='yield'),
 Yield(id='432347ad-eeb1-4007-9fa4-52904db28196', time=datetime.datetime(2026, 9, 1, 11, 31, 51, 979000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('0.35741002'), asset='B2M', type='yield'),
 Yield(id='1ccac04c-9101-42b5-b8b6-0164fb88ca9f', time=datetime.datetime(2026, 8, 31, 11, 32, 52, 850000, tzinfo=datetime.timezone.utc), subaccount=None, amo

In [6]:
async def history(start: datetime | None = None, end: datetime | None = None):
  end = end or datetime.now(timezone.utc)
  start = start or end - timedelta(days=1)
  groups = await asyncio.gather(
    spot_trades(start, end),
    crypto_deposits(start, end),
    crypto_withdrawals(start, end),
    earn_yield(start, end),
  )
  for group in groups:
    for observation in group:
      yield HistoryRecord(
        observations=[observation],
        provenance={'source': 'api', 'service': 'bit2me', 'id': source_id('bit2me')},
      )


end = datetime.now(timezone.utc)
start = end - timedelta(days=1)
[record async for record in history(start, end)]

[HistoryRecord(observations=[Yield(id='80bb3a5e-43b3-44e1-a362-12d95f4a9258', time=datetime.datetime(2026, 9, 4, 11, 35, 41, 871000, tzinfo=datetime.timezone.utc), subaccount=None, amount=Decimal('0.35782066'), asset='B2M', type='yield')], provenance={'id': 'bit2me:2026-09-04T15:52:07.163752+00:00:2358c902-d0f1-4dc4-b6cb-e3dee2bcf509', 'source': 'api', 'service': 'bit2me'})]

## Coverage

**`Snapshots`: fully supported.** `spot_balances()`, `earn_balances()`, and
`pocket_balances()` each map 1:1 onto one endpoint, and `snapshot()` composes them with no
loss. There's no derivatives/margin concept on this venue, so `SubaccountSnapshot.positions`
is always empty -- expected, not a gap. `assets` is accepted but ignored, consistent with
the abstract docstring for a venue that always enumerates every held asset regardless.

**`History`: partially supported.** Spot trades, on-chain deposits/withdrawals, and Earn
reward payouts each map onto their matching `Observation` subtype and are live-tested
above: 10 deposits and 8 withdrawals across four networks in the last year, plus the daily
B2M reward payouts. Three things the mapping cannot fill in, all confirmed against every
`receive`/`send` row this account has (2025 and 2026, both directions):

- **`tx_id` is always `None`.** `v2.wallet.transaction`'s `transaction` object only ever
  carries `confirmedAt`/`confirmationCount`; the `hash` its schema declares is never
  actually returned, so the on-chain hash isn't recoverable from this endpoint.
- **`src_address` is always `None`.** Only one side of a transfer carries
  `address`/`addressNetwork`, and it is the *pocket* side in both directions -- so a
  deposit's `network` has to be read off the destination (the blockchain `origin` has
  neither field), and the sender's address is never reported at all.
- **`fee` is always `None`.** No row in either year carries a `fee` object, so the network
  fee a withdrawal paid is not in the history listing (`v1.wallet.transactions.preview()`,
  used in `wallet.ipynb`, is the only place a withdrawal fee shows up).

The `year` filter is also the only date filter (`v2.wallet.transaction` has no
`startTime`/`endTime` params, unlike `v1.trading.trade`), so both helpers walk
`start.year..end.year` and re-apply the window client-side. Cancelled withdrawals stay in
the listing with the amount they would have sent, so rows whose `status` isn't
`'completed'` are dropped.

**Not covered here, but present on the venue:**
- **Internal transfers** between spot/earn/pocket compartments (`origin.class`/
  `destination.class` of `'pocket'`/`'trading'`/`'earn'`, reachable through the
  `deposit-earn`/`withdrawal-earn`/`deposit-trading`/`withdrawal-trading` operations) --
  would map to `InternalTransfer`, omitted here to keep the demonstrated mapping to the
  sources whose semantics this notebook actually confirmed end-to-end.
- **Fiat deposits/withdrawals** (bank transfer, card/Teller purchases) -- same
  `v2.wallet.transaction` endpoint. `operation='withdrawal'` returns them (this account
  has a `pocket -> bankAccount` row); the matching fiat *deposit* (`subtype='funding'`,
  `method='bank-transfer'`) shows up in the unfiltered listing but is returned by no
  `operation` value at all, which is why the fiat side isn't mapped here.
- **Loans** (`v1.loan.movements`, `v1.loan.orders`) and **Social Pay** (`v1.social_pay`) --
  separate ledgers, not explored here.
- **`crypto_ws`** -- the Crypto API WebSocket (`authenticate`, then the account's own
  notification firehose). `docs/contract/report.yml` declares only `snapshot` and
  `history`, with no streaming method for it to implement, so it is deliberately left
  unmapped.

`v1.wallet.transaction` and `v3.wallet.transaction` both exist as alternatives to
`v2.wallet.transaction`. v1 is marked `@deprecated` in `typed_bit2me` and is gone upstream
too -- calling it returns `404 Cannot GET /v1/wallet/transaction`. v3 works on this
account's credentials and returns identically-shaped rows, but pages by opaque cursor
instead of `offset`, and its listing is not year-scoped (one 100-row page spans 2025 and
2026), so it -- not the year walk above -- is what a real implementation should use.
`v2` is kept here because it is the version whose `operation`/`year` filtering this
notebook mapped against.

No subscribe/redeem/withdraw call exists in the abstract `Report` interface for this
venue, so every call above is read-only and there was nothing to write-but-not-execute.